### Data Modelling 

The data modelling pipeline is as follows
- Load cleaned data
- Define X & y (Independent and dependent variable)
- Split data into train/val (stratified sample)
- Convert text into numeric using TF-IDF
- Train model using Multinomial Logistic regression (baseline model)
- Evaluate model (macro F1)
- Diagnose
- Feature engineering
- Re-train
- Evaluate
- Repeat
- Final test set ((from API))

=======

CBOW vs TF-IDF


In [ ]:
#import sys
#! "{sys.executable}" -m pip install nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nbformat]


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn import linear_model, metrics
import wandb

In [4]:
## Data loading

clean_df = pd.read_csv("/Users/soliufatai/Documents/PersonalDocuments/Data_Science_ML_AI_Krish_Naik/Complete-Data-Science-With-Machine-Learning-And-NLP-2024-main/2-Introduction/Intro/emotion-text-ml/data/cleaned_df.csv")

In [5]:
clean_df.head()

,emotion,clean_text
0,optimism,worry is a down payment on a problem you may n...
1,anger,my roommate its okay that we cant spell becaus...
2,joy,no but thats so cute atsu was probably shy abo...
3,anger,rooneys fucking untouchable isnt he been fucki...
4,sadness,its pretty depressing when u hit pan on ur fav...


In [6]:
## Define X & y

X = clean_df['clean_text']
y = clean_df['emotion']

In [7]:
## Data splitting

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.2, stratify=y, random_state = 30)


In [8]:
type(X_train)

pandas.core.series.Series

In [9]:
## Convert text into numeric using tf-idf

vectorizier = TfidfVectorizer()
X_train_tfidf = vectorizier.fit_transform(X_train)
X_val_tfidf = vectorizier.transform(X_val)

In [10]:
print(X_val_tfidf)

  (np.int32(0), np.int32(535))	0.17802577919490978
  (np.int32(0), np.int32(865))	0.17729821192456163
  (np.int32(0), np.int32(907))	0.21439416999736366
  (np.int32(0), np.int32(2188))	0.5005834769345037
  (np.int32(0), np.int32(3135))	0.2841148363034326
  (np.int32(0), np.int32(3277))	0.13123632608527264
  (np.int32(0), np.int32(3644))	0.24180621759158036
  (np.int32(0), np.int32(4015))	0.30068715893669956
  (np.int32(0), np.int32(4385))	0.17876402020382703
  (np.int32(0), np.int32(4584))	0.19855209827280068
  (np.int32(0), np.int32(5149))	0.3170632362302647
  (np.int32(0), np.int32(5214))	0.29538515162006107
  (np.int32(0), np.int32(6844))	0.306614298842261
  (np.int32(0), np.int32(7069))	0.20080572229230315
  (np.int32(1), np.int32(2087))	0.7977946879768627
  (np.int32(1), np.int32(6507))	0.6029292129561314
  (np.int32(2), np.int32(199))	0.3268680271136798
  (np.int32(2), np.int32(801))	0.35112214493772576
  (np.int32(2), np.int32(876))	0.24494801900135835
  (np.int32(2), np.int32(3

In [11]:
feature_names = vectorizier.get_feature_names_out()

print(feature_names[6568])

to


In [12]:
## Convert the y_train and y_val to numeric using label encoder
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_val_enc = le.transform(y_val)

In [27]:
## Log baseline experiment - experiement 1

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged.
    entity="fataisoliu-fs-saleeh",
    # Set the wandb project where this run will be logged.
    project="emotion-text-ml",
    name="baseline-logreg",
    # Track hyperparameters and run metadata.
    config={
        "model": "Logistic regression",
        "vectorizer": "tfidf",
        "ngram_range": (1, 1),
        "max_iter": 1000
    },
)

In [28]:
# Train baseline model

clf = linear_model.LogisticRegression(max_iter=10000, random_state = 2)
clf.fit(X_train_tfidf, y_train_enc)

y_pred = clf.predict(X_val_tfidf)

accuracy = metrics.accuracy_score(y_val_enc, y_pred) * 100
macro_f1 = metrics.f1_score(y_val_enc, y_pred, average= 'macro')

print(f"Logistic Regression model accuracy: {accuracy:.2f}%")
print(f"Logistic Regression Macro f1_score: {macro_f1}")


## Log results

wandb.log({
    "accuracy": accuracy,
    "macro_f1": macro_f1
})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=y_val_enc,
        preds=y_pred,
        class_names=le.classes_
    )
})

wandb.finish()

Logistic Regression model accuracy: 66.87%
Logistic Regression Macro f1_score: 0.5760688434433853


accuracy,▁
macro_f1,▁
accuracy,66.87117
macro_f1,0.57607


In [21]:
## Match score to classes

f1_score = metrics.f1_score(y_val_enc, y_pred, average=None)

for label, score in zip(le.classes_, f1_score):
    print(f"f1_score by class {label}: {score:.4f}")

f1_score by class anger: 0.7424
f1_score by class joy: 0.5877
f1_score by class optimism: 0.3333
f1_score by class sadness: 0.6408


 The above shows f1_score by class. The f1_score for class optimism is lower compared to class anger, this is as a result of class imbalance.

Next step

- TF-IDF or embeddings
- Apply PCA (quick check)
- Apply t-SNE (main visual)
- Plot points coloured by emotion

Experiment two

- Add n-gram (from unigram ---> bigram)
- 